# Phishing URL Detection System

**IBM SkillsBuild Cyber Security 2026 — Advanced Project**

- **Student:** Rizvanali Asmatali Sheliya
- **Enrollment number:** 246260332035

## Aim
Train and evaluate reproducible machine-learning models to classify phishing versus legitimate website records using the official UCI Phishing Websites dataset.

> **Scope and safety:** This dataset contains 30 pre-computed website/URL-related features. The notebook predicts from those feature columns; it does not visit or scan a live URL and does not claim that a site is safe. Dataset: [https://archive.ics.uci.edu/dataset/327/phishing+websites](https://archive.ics.uci.edu/dataset/327/phishing+websites). DOI: [https://doi.org/10.24432/C51W2X](https://doi.org/10.24432/C51W2X).

## Workflow
1. Load the source dataset and inspect its dimensions, columns, missing values, and target labels.
2. Map labels to `1 = phishing`, `0 = legitimate`.
3. Make a stratified train/test split with a fixed seed.
4. Compare Logistic Regression, Random Forest, and Extra Trees models.
5. Run stratified cross-validation on the training split only.
6. Select the model by cross-validation F1 and evaluate once on the held-out test set.
7. Save the fitted model and evaluation artifacts.

**Reproducibility:** all random states are fixed. Actual scores are printed when the notebook is run; no metrics are hard-coded.

## 1. Install dependencies
Run `pip install -r requirements.txt` in the same environment as Jupyter. The first run needs internet access to retrieve the UCI dataset.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay
)

RANDOM_STATE = 42
TEST_SIZE = 0.20
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries loaded. NumPy:", np.__version__, "| pandas:", pd.__version__)

## 2. Download and inspect the dataset
The official UCI dataset ID is 327. If the service is unavailable, download the dataset from the official page and retry later.

In [ ]:
dataset = fetch_ucirepo(id=327)
X = dataset.data.features.copy()
y_raw = dataset.data.targets.copy()

if isinstance(y_raw, pd.DataFrame):
    if y_raw.shape[1] != 1:
        raise ValueError(f"Expected one target column, received {y_raw.shape[1]}")
    y_raw = y_raw.iloc[:, 0]

print("Feature matrix:", X.shape)
print("Raw target values:")
print(y_raw.value_counts(dropna=False))
display(X.head())
display(dataset.variables.head(10))

## 3. Clean data and encode the target

The original UCI `Result` convention is `-1 = phishing` and `1 = legitimate`. The following cell handles those numeric labels and also accepts clearly named string labels. It fails loudly if it encounters an unexpected target scheme.

In [ ]:
# Make feature values numeric where possible; invalid entries become missing and are imputed in pipelines.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)

def encode_phishing_target(series):
    values = set(pd.Series(series).dropna().astype(str).str.strip().str.lower().unique())
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().all() and set(numeric.unique()).issubset({-1, 1}):
        # UCI source labels: -1 phishing, +1 legitimate
        return (numeric == -1).astype(int), {-1: "phishing", 1: "legitimate"}
    normalized = pd.Series(series).astype(str).str.strip().str.lower()
    phishing_terms = {"-1", "phishing", "phish", "malicious", "bad"}
    legit_terms = {"1", "legitimate", "legit", "benign", "good"}
    if values.issubset(phishing_terms | legit_terms):
        y_encoded = normalized.isin(phishing_terms).astype(int)
        return y_encoded, {"1": "phishing", "0": "legitimate"}
    raise ValueError(f"Unexpected target labels: {sorted(values)}. Inspect source labels before continuing.")

y, label_mapping = encode_phishing_target(y_raw)
y = pd.Series(y, index=X.index, name="is_phishing").astype(int)

if y.nunique() != 2:
    raise ValueError("Target must contain both classes.")
if X.shape[1] < 1:
    raise ValueError("No usable feature columns found.")

print("Target mapping: 1 = phishing, 0 = legitimate")
print("Rows:", len(X), "| Features:", X.shape[1])
print("Missing feature cells:", int(X.isna().sum().sum()))
print("Class counts (0=legitimate, 1=phishing):")
display(y.value_counts().sort_index().rename(index={0:"legitimate (0)", 1:"phishing (1)"}))

## 4. Train/test split
The held-out test data is not used for model selection or cross-validation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print("Train phishing rate:", round(float(y_train.mean()), 4))
print("Test phishing rate:", round(float(y_test.mean()), 4))

## 5. Define candidate models

Each estimator is wrapped in a pipeline that imputes missing values. Logistic Regression also uses standard scaling. Tree ensembles can capture nonlinear feature interactions. Class labels are defined consistently: class 1 means phishing.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_STATE
        ))
    ]),
    "Extra Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_STATE
        ))
    ]),
}
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 6. Cross-validation on training data
Selection uses mean CV F1. This helps avoid selecting a model based on the held-out test results.

In [ ]:
cv_rows = []
for name, estimator in models.items():
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=-1, return_train_score=False, error_score="raise"
    )
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = float(np.mean(scores[f"test_{metric}"]))
        row[f"{metric}_std"] = float(np.std(scores[f"test_{metric}"], ddof=1))
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("f1_mean", ascending=False).reset_index(drop=True)
display(cv_results.round(4))
best_name = str(cv_results.loc[0, "model"])
best_model = models[best_name]
print("Selected by mean 5-fold training CV F1:", best_name)

## 7. Fit selected model and evaluate once on test set

In [ ]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_score = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "selected_model": best_name,
    "test_rows": int(len(y_test)),
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision_phishing": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall_phishing": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_phishing": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_score)),
}
print(json.dumps(test_metrics, indent=2))
print("\nClassification report (class 1 = phishing):")
print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["Legitimate (0)", "Phishing (1)"], zero_division=0
))

## 8. Confusion matrix and ROC curve
False negatives are phishing records predicted as legitimate. In a practical anti-phishing system, that error can be especially costly.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted legitimate", "Predicted phishing"],
    yticklabels=["Actual legitimate", "Actual phishing"]
)
plt.title("Held-out test confusion matrix")
plt.ylabel("Actual class")
plt.xlabel("Predicted class")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_score)
plt.title(f"ROC curve — {best_name}")
plt.tight_layout()
plt.show()

## 9. Feature importance (when supported)
Tree models expose impurity-based feature importance. Treat this as a model diagnostic, not causal proof. Logistic Regression uses coefficient magnitude instead.

In [ ]:
final_estimator = best_model.named_steps["model"]
if hasattr(final_estimator, "feature_importances_"):
    importance_values = final_estimator.feature_importances_
    importance_label = "Feature importance"
elif hasattr(final_estimator, "coef_"):
    importance_values = np.abs(final_estimator.coef_[0])
    importance_label = "Absolute coefficient"
else:
    importance_values = None

if importance_values is not None:
    importance = pd.Series(importance_values, index=X.columns).sort_values(ascending=False).head(15)
    display(importance.rename(importance_label).to_frame().round(4))
    importance.sort_values().plot(kind="barh", figsize=(8, 6), title=f"Top 15 features — {importance_label}")
    plt.xlabel(importance_label)
    plt.tight_layout()
    plt.show()
else:
    print("Selected estimator does not expose standard feature importance.")

## 10. Save reproducible artifacts
The model is saved with its feature column order and test metrics. Load joblib artifacts only from sources you trust.

In [ ]:
joblib.dump(
    {"model": best_model, "feature_columns": list(X.columns),
     "target_mapping": {"1": "phishing", "0": "legitimate"},
     "dataset_id": 327, "random_state": RANDOM_STATE},
    ARTIFACT_DIR / "best_phishing_model.joblib"
)
cv_results.to_csv(ARTIFACT_DIR / "cross_validation_results.csv", index=False)
pd.DataFrame([test_metrics]).to_json(ARTIFACT_DIR / "test_metrics.json", orient="records", indent=2)
pd.DataFrame(cm, index=["actual_legitimate", "actual_phishing"],
             columns=["predicted_legitimate", "predicted_phishing"]).to_csv(
    ARTIFACT_DIR / "confusion_matrix.csv"
)
print("Saved artifacts to:", ARTIFACT_DIR.resolve())
print("Saved files:", sorted(p.name for p in ARTIFACT_DIR.iterdir()))

## 11. Discussion and limitations

- **Recall vs precision:** high phishing recall reduces missed phishing records, but may increase false alarms.
- **Dataset age:** this dataset was donated in 2015. Real phishing patterns evolve quickly, so scores should not be assumed to transfer to current traffic.
- **Potential leakage / dependence:** random stratified splitting may place related sites or collection patterns across train and test. A domain-grouped or time-based external evaluation is a stronger follow-up.
- **Feature availability:** the dataset includes pre-computed site/URL features. This notebook does not derive all 30 features from an arbitrary URL.
- **Operational safety:** do not open suspected phishing URLs or enter credentials. A model prediction is a signal, not proof that a page is safe.

## References

1. Mohammad, R., & McCluskey, L. (2012). *Phishing Websites* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C51W2X
2. UCI dataset page: https://archive.ics.uci.edu/dataset/327/phishing+websites